In [1]:
# Imports
import os
from pathlib import Path
import json
import torch
from faster_whisper import WhisperModel
from pydub import AudioSegment
from nemo.collections.asr.models import SortformerEncLabelModel

# File section and model size
audio_path = Path(r"C:\Users\somlab\Downloads\audio1365983309.m4a")
whisper_size = "small" # tiny / base / small / medium / large-v3 (small fits on my Nvidia 1080 GPU)

# Check if CUDA is installed properly and update settings. Cannot do diarization without CUDA.
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "auto" if device == "cuda" else "int8" # Some systems to float16 or float32, auto lets it do either
print(f"Using device: {device} ({compute_type})")

c:\Users\somlab\miniconda3\envs\bb_voice_transcription\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-07-17 08:52:13 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
W0717 08:52:13.786000 4404 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Using device: cuda (auto)


### Raw transcription
Transcribe the audio file aithout attempting to identify the speaker. On my computer this took ~15 minutes for a 1 hour interview with CPU only and ~3 minutes with GPU.

Note that GPU requires CUDA 12x.

We can use this whole file transcript to chunk the audio into suitable sections.

In [ ]:
# Initialize the model (model size, device [cuda or cpu], compute_type).
# word_timestamps=True gives per-word times needed for speaker assignment.
model = WhisperModel(whisper_size, device=device, compute_type=compute_type)
segments, info = model.transcribe(str(audio_path), beam_size=5, word_timestamps=True)

# transcribe() returns a generator; materialize it so we can reuse the segments.
segments = list(segments)
print("Detected language '%s' with probability %f" % (info.language, info.language_probability))
for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

### Speaker diarization
Use NVIDIA NeMo's **Sortformer** model to tag a speaker to each word segment.
Sortformer requires **mono 16 kHz WAV** input, so we first re-export the audio.

In [2]:
# --- 1. Convert audio to mono 16 kHz WAV (required by NeMo) ---
wav_path = audio_path.with_suffix(".16k.wav")
AudioSegment.from_file(audio_path).export(
    wav_path, format="wav", parameters=["-ac", "1", "-ar", "16000"]
)
print("Wrote", wav_path)

Wrote C:\Users\somlab\Downloads\audio1365983309.16k.wav


Create a config file for the diarizer. 

In [3]:
# Dump a config
manifest_path = audio_path.with_suffix(".json")
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write(json.dumps({"audio_filepath": str(wav_path), "offset": 0, "duration": 100000, "label": "infer", "text": "-"}) + "\n")

# Diarize
print("Running NeMo Sortformer diarization...")
diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
diar_model.eval()

# diarize() returns one list of "start end speaker_label" strings per input file.
prediction = diar_model.diarize(audio=[str(manifest_path)], batch_size=1)

speaker_turns = []
for line in prediction[0]:
    start, end, speaker = line.split()
    speaker_turns.append({"start": float(start), "end": float(end), "speaker": speaker})

speakers = sorted({t["speaker"] for t in speaker_turns})
print(f"{len(speaker_turns)} speaker turns across {len(speakers)} speakers: {speakers}")

Running NeMo Sortformer diarization...


[NeMo W 2026-07-17 08:52:57 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    num_buckets: 10
    bucket_duration_bins:
    - 10
    - 20
    - 30
    - 40
    - 50
    - 60
    - 70
    - 80
    - 90
    pin_memory: true
    min_duration: 80
    max_duration: 90
    batch_duration: 400
    quadratic_duration: 1200
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-07-17 08:52:57 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or

[NeMo I 2026-07-17 08:52:58 save_restore_connector:285] Model SortformerEncLabelModel was successfully restored from C:\Users\somlab\.cache\huggingface\hub\models--nvidia--diar_sortformer_4spk-v1\snapshots\9f17b10df44c0a4c8f3c86fbddc9ee2d6ab9ac08\diar_sortformer_4spk-v1.nemo.
[NeMo I 2026-07-17 08:52:58 vad_utils:81] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


[NeMo W 2026-07-17 08:52:58 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: session_len_sec,soft_label_thres,num_spks
Diarizing: 0it [00:01, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Merge transcript with speakers
For each Whisper segment we look at its words, find which diarization speaker each
word overlaps most in time, and take the majority speaker for the segment.

In [ ]:
# --- 3. Assign a speaker to each transcript segment ---
def _speaker_for(start, end):
    """Speaker whose turns overlap the [start, end] interval the most."""
    best, best_overlap = "unknown", 0.0
    for t in speaker_turns:
        overlap = max(0.0, min(end, t["end"]) - max(start, t["start"]))
        if overlap > best_overlap:
            best_overlap, best = overlap, t["speaker"]
    return best


def assign_speaker(seg):
    """Majority speaker over a segment's words (falls back to segment span)."""
    votes = {}
    for w in (seg.words or []):
        spk = _speaker_for(w.start, w.end)
        votes[spk] = votes.get(spk, 0.0) + (w.end - w.start)
    return max(votes, key=votes.get) if votes else _speaker_for(seg.start, seg.end)


transcript = [
    {"start": s.start, "end": s.end, "speaker": assign_speaker(s), "text": s.text.strip()}
    for s in segments
]

# Print, grouping consecutive segments from the same speaker.
last = None
for row in transcript:
    if row["speaker"] != last:
        print(f"\n[{row['speaker']}]")
        last = row["speaker"]
    print(f"  ({row['start']:.1f}-{row['end']:.1f}) {row['text']}")

In [ ]:
# --- 4. Save the speaker-annotated transcript to a text file ---
out_path = audio_path.with_suffix(".transcript.txt")
with open(out_path, "w", encoding="utf-8") as f:
    last = None
    for row in transcript:
        if row["speaker"] != last:
            f.write(f"\n[{row['speaker']}]\n")
            last = row["speaker"]
        f.write(f"({row['start']:.1f}-{row['end']:.1f}) {row['text']}\n")
print("Saved transcript to", out_path)